# Weird AI Lesson 7: Instruction Fine-Tuning

In this notebook, Weird AI evolves from a lyric generator into a **parody assistant**.

Earlier lessons trained a model to predict the next token. Instruction fine-tuning changes the goal:

```text
Instruction + optional input
        ↓
Model
        ↓
Useful response
```

By the end of this notebook, you should understand instruction formatting, instruction datasets, custom batching, shifted targets, and why padding targets are replaced with `-100`.

## Section 1: Pretraining vs Instruction Fine-Tuning

Pretraining teaches the model to continue text.

Instruction fine-tuning teaches the model to respond to a task.

For Weird AI, this means moving from raw lyric continuation to instructions such as:

```text
Write a silly parody chorus about computer networking.
```

In [1]:
import json
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(123)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Section 2: Create a Tiny Weird AI Instruction Dataset

A real instruction-tuning dataset would have many examples. This tiny dataset is only for learning the workflow.

In [2]:
instruction_data = [
    {
        "instruction": "Write a silly parody chorus about computer networking.",
        "input": "",
        "output": "Packets in the night, routers glowing bright, I lost my ping but I still feel alright."
    },
    {
        "instruction": "Rewrite the lyric idea as a funny parody.",
        "input": "I miss you every night under the stars.",
        "output": "I miss my Wi-Fi every night under the bars."
    },
    {
        "instruction": "Write a short parody lyric about debugging Java.",
        "input": "",
        "output": "I chased a null through the break of dawn, but the stack trace kept singing on."
    },
    {
        "instruction": "Turn this serious lyric idea into a silly food parody.",
        "input": "My heart is broken and I cannot sleep.",
        "output": "My tart is broken and my nachos weep."
    },
]

print(f"Number of examples: {len(instruction_data)}")
print(instruction_data[0])

Number of examples: 4
{'instruction': 'Write a silly parody chorus about computer networking.', 'input': '', 'output': 'Packets in the night, routers glowing bright, I lost my ping but I still feel alright.'}


## Section 3: Format an Instruction Example

We will use an Alpaca-style format with `### Instruction:`, optional `### Input:`, and `### Response:` sections.

In [3]:
def format_input(entry):
    instruction_text = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = (
        f"\n\n### Input:\n{entry['input']}"
        if entry["input"]
        else ""
    )

    return instruction_text + input_text


example = instruction_data[1]
model_input = format_input(example)
desired_response = f"\n\n### Response:\n{example['output']}"

print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Rewrite the lyric idea as a funny parody.

### Input:
I miss you every night under the stars.

### Response:
I miss my Wi-Fi every night under the bars.


## Section 4: Save and Reload Instruction Data

TODO:

1. Save the dataset to `../data/instruction/weird_ai_instruction_data.json`
2. Load it back into `loaded_data`
3. Print the number of examples

In [4]:
data_path = Path("../data/instruction/weird_ai_instruction_data.json")
data_path.parent.mkdir(parents=True, exist_ok=True)

# TODO: save instruction_data using json.dump(..., indent=4)

json.dump(instruction_data, data_path.open("w"), indent=4)
# TODO: load the JSON file back into loaded_data

loaded_data = json.load(data_path.open("r"))

print(f"Loaded examples: {len(loaded_data)}")

Loaded examples: 4


## Section 5: Tokenize the Formatted Examples

The model reads token IDs, not raw strings. For this lesson, use `SimpleCharacterTokenizer`.

In [5]:
from weird_ai.tokenizer import SimpleCharacterTokenizer

all_text = "\n".join(
    format_input(entry) + f"\n\n### Response:\n{entry['output']}"
    for entry in instruction_data
)

tokenizer = SimpleCharacterTokenizer(all_text)

print(f"Vocabulary size: {len(tokenizer.chars)}")
print(tokenizer.chars)

Vocabulary size: 39
['\n', ' ', '#', ',', '-', '.', ':', 'B', 'F', 'I', 'J', 'M', 'P', 'R', 'T', 'W', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'y']


## Section 6: Build an Instruction Dataset

The dataset should format each instruction-response example, tokenize it, and return one list of token IDs.

In [6]:
class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer
        self.encoded_texts = []

        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text

            # TODO: encode full_text and append it to self.encoded_texts
            encoded = self.tokenizer.encode(full_text)
            self.encoded_texts.append(encoded)

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)


instruction_dataset = InstructionDataset(instruction_data, tokenizer)

# TODO: print the first encoded item and its length
print(f"First encoded item: {instruction_dataset[0]}")
print(f"Length of first encoded item: {len(instruction_dataset[0])}")

First encoded item: [7, 20, 26, 29, 37, 1, 24, 33, 1, 16, 28, 1, 24, 28, 33, 34, 32, 35, 18, 34, 24, 29, 28, 1, 34, 23, 16, 34, 1, 19, 20, 33, 18, 32, 24, 17, 20, 33, 1, 16, 1, 34, 16, 33, 25, 5, 1, 15, 32, 24, 34, 20, 1, 16, 1, 32, 20, 33, 30, 29, 28, 33, 20, 1, 34, 23, 16, 34, 1, 16, 30, 30, 32, 29, 30, 32, 24, 16, 34, 20, 26, 38, 1, 18, 29, 27, 30, 26, 20, 34, 20, 33, 1, 34, 23, 20, 1, 32, 20, 31, 35, 20, 33, 34, 5, 0, 0, 2, 2, 2, 1, 9, 28, 33, 34, 32, 35, 18, 34, 24, 29, 28, 6, 0, 15, 32, 24, 34, 20, 1, 16, 1, 33, 24, 26, 26, 38, 1, 30, 16, 32, 29, 19, 38, 1, 18, 23, 29, 32, 35, 33, 1, 16, 17, 29, 35, 34, 1, 18, 29, 27, 30, 35, 34, 20, 32, 1, 28, 20, 34, 37, 29, 32, 25, 24, 28, 22, 5, 0, 0, 2, 2, 2, 1, 13, 20, 33, 30, 29, 28, 33, 20, 6, 0, 12, 16, 18, 25, 20, 34, 33, 1, 24, 28, 1, 34, 23, 20, 1, 28, 24, 22, 23, 34, 3, 1, 32, 29, 35, 34, 20, 32, 33, 1, 22, 26, 29, 37, 24, 28, 22, 1, 17, 32, 24, 22, 23, 34, 3, 1, 9, 1, 26, 29, 33, 34, 1, 27, 38, 1, 30, 24, 28, 22, 1, 17, 35, 34, 1, 9

## Section 7: Implement a Custom Collate Function

The collate function should:

1. Find the longest example in the batch
2. Pad shorter examples
3. Create input token IDs
4. Create shifted target token IDs
5. Replace extra padding targets with `-100`

In [11]:
def custom_collate_fn(
    batch,
    pad_token_id=0,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    # TODO: find maximum length in batch, adding 1 for the appended pad/end token
    batch_max_length = max(len(item) for item in batch) + 1

    inputs_lst = []
    targets_lst = []

    for item in batch:
        new_item = item.copy()

        # TODO: append one pad_token_id to new_item
        new_item.append(pad_token_id)

        # TODO: pad new_item to batch_max_length
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))

        # TODO: inputs are padded[:-1], targets are padded[1:]
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        # TODO: replace all but the first pad_token_id in targets with ignore_index
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        # TODO: optionally truncate inputs and targets to allowed_max_length
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

## Section 8: Test the Collate Function

Use small fake token lists so you can clearly see input/target shifting and masking.

In [12]:
sample_batch = [
    [1, 2, 3, 4],
    [5, 6],
    [7, 8, 9],
]

# TODO: after implementing custom_collate_fn, uncomment this code:
inputs, targets = custom_collate_fn(sample_batch, pad_token_id=0, device=device)
print("Inputs:")
print(inputs)
print("\nTargets:")
print(targets)

Inputs:
tensor([[1, 2, 3, 4],
        [5, 6, 0, 0],
        [7, 8, 9, 0]], device='cuda:0')

Targets:
tensor([[   2,    3,    4,    0],
        [   6,    0, -100, -100],
        [   8,    9,    0, -100]], device='cuda:0')


## Section 9: Create a DataLoader

Use `functools.partial` to preconfigure the custom collate function.

In [13]:
from functools import partial

customized_collate_fn = partial(
    custom_collate_fn,
    pad_token_id=0,
    ignore_index=-100,
    allowed_max_length=256,
    device=device
)

# TODO: create a DataLoader using instruction_dataset and customized_collate_fn
instruction_loader = DataLoader(
    instruction_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=customized_collate_fn
)

# TODO: inspect one batch
inputs, targets = next(iter(instruction_loader))
print(inputs.shape)
print(targets.shape)

torch.Size([2, 256])
torch.Size([2, 256])


## Section 10: Understanding `-100`

PyTorch cross entropy ignores target values of `-100` by default. That lets us prevent padding tokens from affecting the loss.

In [14]:
import torch.nn.functional as F

logits = torch.tensor([
    [-1.0, 1.0],
    [-0.5, 1.5],
    [-0.5, 1.5],
])

targets_with_padding = torch.tensor([0, 1, -100])

loss = F.cross_entropy(logits, targets_with_padding)

print(loss)

tensor(1.1269)


## Section 11: Connect to Weird AI Fine-Tuning

Instruction fine-tuning uses the same core training loop as pretraining, but the examples are formatted as instruction-response pairs.

In [15]:
print("Instruction fine-tuning uses next-token prediction on formatted instruction-response examples.")

Instruction fine-tuning uses next-token prediction on formatted instruction-response examples.


## Section 12: Reflection Questions

Answer these questions in your submitted Word document.

1. How is instruction fine-tuning different from pretraining?
Pretraining typically is where we feed the model lots of language and information without labels to let it learn grammar and general language.
Fine-tuning is done afterwards to make the llm fit its specific needs. So for a classifier this includes labels with data to train on, and 
for a prompt-response type of LLM we need to feed it data that has sections that are labeled with prompt, response, and any other inputs.
2. Why do instruction datasets contain instruction/input/output fields?
So that the model can learn the format of prompt and response and learn what a proper response is to specific examples and then it can 
learn to generalize.
3. Why do we format examples with sections like `### Instruction:` and `### Response:`?
This is a structured way to teach the LLM, this way every training set has a guaranteed way for the LLM to know where instruction begins
and response begins.
4. Why do batches need padding?
So that all training sets have the same length for processing.
5. Why are some target tokens replaced with `-100`?
Cross entropy ignores -100, so that the model does not learn from padding tokens.
6. How does instruction fine-tuning move Weird AI from lyric generator to parody assistant?
Previously it was just predicting next tokens until it filled a specific length of tokens. With fine tuning it can now understand a prompt and what a 
typical response would be. 
7. What kinds of instruction examples would make Weird AI better at parody generation?
A wide variety of training with varied genre and lyric requests. This way it does not overfit and copy songs to closely.